# Parallel 多 Agent 并行基线

该基线对每个问题都并行运行三个 Agent，然后汇总它们的回复。

## 与 PHA 对比

| 比较项 | PHA 编排架构 | Parallel 并行基线 |
| --- | --- | --- |
| 路由 | 选择合适的 Agent | 每次运行三个 Agent |
| Agent 层级 | 设有主 Agent 和辅助 Agent | 各 Agent 地位相同 |
| 结果汇总 | 在编排中协调各方分析 | 由大模型直接汇总 |
| 调用范围 | 按需调用 | 固定调用全部三个 Agent |

这组对照用于评估 PHA 中按需路由的价值。

## 环境搭建

先设置环境并加载所需模块。

In [ ]:
import os
import sys

# 将父目录添加到导入路径
sys.path.insert(0, os.path.abspath('..'))

# 设置 API 密钥( 替换为您自己的或设定的环境变量)
# os.environ ['GEMINI_API_KEY']="这里的钥匙"
# os.environ ['TAVILY_API_KEY'] = '你的钥匙 -这里'#可选，用于网页搜索

In [ ]:
from config.settings import Settings
from pha.utils import load_persona
from pha.agents import (
    DataScienceAgent,
    DomainExpertAgent,
    HealthCoachAgent,
    ParallelMultiAgentBaseline,
    create_parallel_baseline,
)

# 装入设置和 API 密钥
settings = Settings()
GEMINI_API_KEY = settings.gemini_api_key
TAVILY_API_KEY = settings.tavily_api_key

print(f"Gemini API key configured: {bool(GEMINI_API_KEY)}")
print(f"Tavily API key configured: {bool(TAVILY_API_KEY)}")

## 加载样本数据

加载样本用户的数据，供 Agent 分析。

In [ ]:
# 装入样本人a 数据
summary_df, activities_df, profile_df, population_df = load_persona(settings=settings)

print(f"Summary data: {len(summary_df)} days")
print(f"Activities: {len(activities_df)} records")
print(f"Profile: {profile_df['age'].values[0]} year old {profile_df['gender'].values[0]}")

## 方法 1：通过工厂函数快速配置

下面直接创建已配置好的并行基线。

In [ ]:
# 创建并行 基线 与工厂函数
baseline = create_parallel_baseline(
    gemini_api_key=GEMINI_API_KEY,
    tavily_api_key=TAVILY_API_KEY,
    settings=settings,
    debug_verbose=True,
)

print("Parallel baseline created successfully!")

## 方法 2：手动配置

逐个配置可以更细致地控制 Agent 的设置。

In [ ]:
# # 手动设置( 无法使用而非工厂功能)
# 
# # 1 创建 数据科学 Agent
# ds_agent = DataScienceAgent(settings=settings)
# ds_agent.configure(gemini_api_key=GEMINI_API_KEY)
# 
# # 2 创建 领域专家 Agent
# de_agent = DomainExpertAgent(tavily_api_key=TAVILY_API_KEY)
# de_agent.get_agent(gemini_api_key=GEMINI_API_KEY)
# 
# # 3 创建 健康教练 Agent
# coach_agent = HealthCoachAgent(simple_mode=True)
# coach_agent.configure(gemini_api_key=GEMINI_API_KEY)
# 
# # 4. 创建和配置 基线 (中文(简体) ).
# baseline = ParallelMultiAgentBaseline(debug_verbose=True)
# baseline.configure(gemini_api_key=GEMINI_API_KEY)
# 基线 . set_agents (
#     data_science_agent=ds_agent,
#     domain_expert_agent=de_agent,
#     health_coach_agent=coach_agent,
# )

## 测试问题 1：睡眠分析

用一个适合从三个角度共同分析的睡眠问题来测试。

In [ ]:
query1 = "How has my sleep been over the past two weeks? Am I getting enough deep sleep?"

print(f"Query: {query1}")
print("=" * 80)
print("\nProcessing... (all 3 agents running in parallel)\n")

response1 = baseline.respond(query1)

print("\n" + "=" * 80)
print("SYNTHESIZED RESPONSE:")
print("=" * 80)
print(response1)

### 查看各 Agent 的回复

可以分别查看每个 Agent 提供了哪些信息。

In [ ]:
individual = baseline.get_individual_responses()

print("=" * 80)
print("DATA SCIENCE AGENT RESPONSE:")
print("=" * 80)
print(individual['data_science'][:2000] + "..." if len(individual['data_science']) > 2000 else individual['data_science'])

In [ ]:
print("=" * 80)
print("DOMAIN EXPERT AGENT RESPONSE:")
print("=" * 80)
print(individual['domain_expert'][:2000] + "..." if len(individual['domain_expert']) > 2000 else individual['domain_expert'])

In [ ]:
print("=" * 80)
print("HEALTH COACH AGENT RESPONSE:")
print("=" * 80)
print(individual['health_coach'][:2000] + "..." if len(individual['health_coach']) > 2000 else individual['health_coach'])

## 测试问题 2：心血管健康

尝试提出一个关于心血管健康的问题。

In [ ]:
# 重置对话以重新开始
baseline.reset_conversation()

query2 = "What does my resting heart rate tell me about my cardiovascular fitness?"

print(f"Query: {query2}")
print("=" * 80)

response2 = baseline.respond(query2)

print("\nSYNTHESIZED RESPONSE:")
print("=" * 80)
print(response2)

## 测试问题 3：活动建议

这个问题可能更适合由健康教练处理。

In [ ]:
baseline.reset_conversation()

query3 = "I want to improve my overall fitness. What should I focus on based on my data?"

print(f"Query: {query3}")
print("=" * 80)

response3 = baseline.respond(query3)

print("\nSYNTHESIZED RESPONSE:")
print("=" * 80)
print(response3)

## 对比要点

- **Parallel** 架构简单，每次都能汇集三个 Agent 的结果，也省去了路由判断。
- **PHA** 按需调用 Agent，由主 Agent 推进对话，协调各部分的回答，并在多轮交互中保留上下文。

这一组对照要回答的问题是，按需路由能否带来比固定并行调用更多的收益？

## 清理

In [ ]:
# 为下次会议重置
baseline.reset_conversation()
print("Conversation reset. Baseline ready for new queries.")